<img src="https://data.source.coop/taco/3dclouds/images/valencia.png" height="45">&nbsp;&nbsp;&nbsp;
<img src="https://data.source.coop/taco/3dclouds/images/oxford.png" height="45">&nbsp;&nbsp;&nbsp;
<img src="https://data.source.coop/taco/3dclouds/images/athens.png" height="45">&nbsp;&nbsp;&nbsp;
<img src="https://data.source.coop/taco/3dclouds/images/harvard.png" height="45">&nbsp;&nbsp;&nbsp;
<img src="https://data.source.coop/taco/3dclouds/images/esa.png" height="45">&nbsp;&nbsp;&nbsp;
<img src="https://data.source.coop/taco/3dclouds/images/canada.jpg" height="45">

---

# Cloud3DTACO — A Global, AI-Ready Dataset for 3D Cloud Reconstruction

*Under review*

**Cesar Aybar**¹&nbsp;&nbsp;·&nbsp;&nbsp;**Shirin Ermis**²&nbsp;&nbsp;·&nbsp;&nbsp;**Lilli Freischem**²&nbsp;&nbsp;·&nbsp;&nbsp;**Stella Girtsou**³⁴&nbsp;&nbsp;·&nbsp;&nbsp;**Kyriaki-Margarita Bintsi**⁵&nbsp;&nbsp;·&nbsp;&nbsp;**Emiliano Diaz Salas-Porras**¹&nbsp;&nbsp;·&nbsp;&nbsp;**Michael Eisinger**⁶&nbsp;&nbsp;·&nbsp;&nbsp;**William Jones**²&nbsp;&nbsp;·&nbsp;&nbsp;**Anna Jungbluth**⁶&nbsp;&nbsp;·&nbsp;&nbsp;**Benoit Tremblay**⁷

<sub>¹ Universitat de València &nbsp;|&nbsp; ² University of Oxford &nbsp;|&nbsp; ³ National Observatory of Athens &nbsp;|&nbsp; ⁴ National Technical University of Athens &nbsp;|&nbsp; ⁵ Harvard Medical School & Massachusetts General Hospital &nbsp;|&nbsp; ⁶ European Space Agency &nbsp;|&nbsp; ⁷ Environment and Climate Change Canada</sub>

---

> **Abstract.** Cloud3DTACO is an AI-ready dataset for global 3D cloud reconstruction, built on the **TACO** (Transparent Access to Cloud-Optimized datasets) specification. It pairs 2D multispectral geostationary imagery from **GOES-16**, **Himawari-8/9**, and **MSG/SEVIRI** with co-located 3D vertical cloud property profiles from **CloudSat** radar — covering the Americas, Asia-Oceania, Europe, and Africa. A dedicated benchmark subset targets **tropical cyclones** using IBTrACS storm tracks. The dataset supports self-supervised pre-training, supervised fine-tuning, and model evaluation workflows under FAIR principles.

---

### `Demo 1` — Self-Supervised Pretraining

This demo walks through a complete **self-supervised pretraining** pipeline on the Cloud3DTACO pretraining split — no CloudSat labels required. The goal is to learn a general-purpose visual representation of geostationary cloud imagery across three satellites simultaneously.

The model is a **UNet autoencoder**: it takes a 17-channel input patch (15 spectral/angle bands + 2 time encoding channels) and learns to reconstruct the 11 core spectral bands (reflectances + brightness temperatures, excluding geometry angles and time). By forcing a compressed bottleneck to reconstruct the input, the encoder learns meaningful cloud features that can later be fine-tuned for 3D profile prediction.

The full pipeline covered in this demo:

1. **Install dependencies** — all packages needed to run the pipeline
2. **Configuration** — band mappings, paths, and training hyperparameters
3. **Transforms** — crop, normalize, and time-encode raw geostationary patches
4. **Dataset & DataLoader** — stream COGs directly from source.coop via `tacoreader`
5. **Model** — UNet autoencoder built with `segmentation-models-pytorch`
6. **Training** — PyTorch Lightning trainer with checkpointing and CSV logging
7. **Results** — loss curve inspection after training


## 1. Install dependencies


In [1]:
%%capture
!pip install tacoreader rasterio numpy pandas segmentation-models-pytorch lightning

## 2. Configuration

Cloud3DTACO unifies three geostationary sensors — **GOES-16** (Americas), **Himawari-8/9** (Asia-Oceania), and **MSG/SEVIRI** (Europe-Africa) — into a single sensor-agnostic 15-band representation. Each sensor has a different native band ordering, so `BAND_MAPPING` remaps each one to the common order defined in `BAND_NAMES`.

The 15 common bands span visible/NIR reflectances, shortwave IR, water vapour channels, thermal IR windows, CO₂ absorption, and solar/satellite geometry angles. At training time, 2 additional channels encoding the **acquisition time** (fraction of year, fraction of day) are appended, giving the model **17 input channels** in total. The reconstruction target is the first **11 bands** only — spectral bands, excluding angles and time.


In [4]:
BAND_MAPPING = {
    "GOES":     [1, 2, 4, 6, 7, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19],
    "HIMAWARI": [2, 3, 4, 6, 7, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19],
    "MSG":      [9, 10, 0, 1, 2, 3,  4,  5,  6,  7,  8, 11, 12, 13, 14],
}

BAND_NAMES = [
    "red", "nir", "swir_1.6", "swir_3.9",
    "wv_6.2", "wv_7.3", "cloud_phase", "ozone",
    "ir_10.5", "ir_11.2", "co2",
    "sat_zen", "sat_azi", "sol_zen", "sol_azi",
]

BAND_TYPES = ["nr"] * 3 + ["bt"] * 8 + ["angle_zen", "angle_azi", "angle_zen", "angle_azi"]

N_COMMON_BANDS = len(BAND_NAMES)                # 15 spectral bands (sensor-agnostic)
N_TIME_BANDS   = 2                              # fraction_of_year, fraction_of_day
N_TOTAL_BANDS  = N_COMMON_BANDS + N_TIME_BANDS  # 17 — model input
N_RECON_BANDS  = 11                             # bands to reconstruct (no angles, no time)

SATELLITES   = list(BAND_MAPPING.keys())
DATASET_DIR  = "https://data.source.coop/taco/3dclouds/pretraining/"
OUTPUT_DIR   = "./output/"
BATCH_SIZE   = 4
NUM_WORKERS  = 2
VAL_FRACTION = 0.05
PATCH_SIZE   = [128, 128]
LR           = 1e-5
WEIGHT_DECAY = 1e-4
ENCODER      = "mobilenet_v2"

## 3. Transforms

Raw geostationary data comes in two physical units: **reflectances** (visible/NIR/SWIR, `nr`) and **brightness temperatures** in Kelvin (thermal/WV bands, `bt`). These have very different numerical ranges and cannot be fed directly to a network. Three transforms are applied in sequence:

- **`RandomCropTransform`** — cuts a 128×128 patch from the full 512×512 scene. `center_crop` mode samples from a small window around the scene centre, avoiding the noisy edges common in geostationary imagery.
- **`MinMaxNormaliseTransform`** — clips and rescales reflectances and brightness temperatures to `[-1, 1]`. Geometry angles are normalized to `[0, 1]`.
- **`TimeTo2DTransform`** — encodes the acquisition timestamp as two constant 2D channels (fraction of year and fraction of day) and concatenates them to the image, expanding from 15 → 17 channels. This lets the model implicitly learn seasonal and diurnal cloud patterns.


In [5]:
import numpy as np
from random import randint


class RandomCropTransform:
    def __init__(self, patch_size, center_crop=False, radius=0):
        self.patch_size  = patch_size
        self.center_crop = center_crop
        self.radius      = radius

    def __call__(self, d):
        arr = d["image"]
        H, W = arr.shape[1], arr.shape[2]
        ph, pw = self.patch_size
        if self.center_crop:
            cx = randint(H // 2 - self.radius, H // 2 + self.radius)
            cy = randint(W // 2 - self.radius, W // 2 + self.radius)
            x0, y0 = cx - ph // 2, cy - pw // 2
        else:
            x0 = randint(0, H - ph)
            y0 = randint(0, W - pw)
        d["image"] = arr[:, x0:x0+ph, y0:y0+pw]
        return d


class MinMaxNormaliseTransform:
    def __init__(self, bt_min=180, bt_max=350, nr_min=0, nr_max=100):
        self.bt_min, self.bt_max = bt_min, bt_max
        self.nr_min, self.nr_max = nr_min, nr_max

    def _minmax(self, x, lo, hi):
        return (np.clip(x, lo, hi) - lo) / (hi - lo) * 2 - 1

    def __call__(self, d):
        arr = d["image"]
        for i, t in enumerate(BAND_TYPES):
            if t == "bt":
                arr[i] = self._minmax(arr[i], self.bt_min, self.bt_max)
            elif t == "nr":
                arr[i] = self._minmax(arr[i], self.nr_min, self.nr_max)
            elif t == "angle_zen":
                arr[i] = np.clip(arr[i], 0, 180) / 180
            elif t == "angle_azi":
                arr[i] = np.clip(arr[i], 0, 360) / 360
        d["image"] = arr
        return d


class TimeTo2DTransform:
    def __init__(self, patch_size):
        self.h, self.w = patch_size

    def __call__(self, d):
        dt  = d["date"].to_pydatetime()
        foy = np.clip(int(dt.strftime("%j")) / 365, 0, 1)
        fod = np.clip(dt.hour / 24 + dt.minute / 1440 + dt.second / 86400, 0, 1)
        time_2d    = np.empty((2, self.h, self.w), dtype=np.float32)
        time_2d[0] = foy
        time_2d[1] = fod
        d["image"] = np.concatenate([d["image"], time_2d], axis=0)
        return d


def build_transform(patch_size, center_crop=True, radius=32):
    crop   = RandomCropTransform(patch_size, center_crop, radius)
    norm   = MinMaxNormaliseTransform()
    time2d = TimeTo2DTransform(patch_size)
    def transform(d):
        d = crop(d)
        d = norm(d)
        d = time2d(d)
        d["image"] = np.nan_to_num(d["image"], nan=0.0)
        d["date"]  = str(d["date"])
        return d
    return transform

## 4. Dataset & DataLoader

`tacoreader.load()` reads the TACO index directly from source.coop and returns a structured table of all available samples. `.flatten()` expands the hierarchical TACO tree to a flat row-per-sample DataFrame. We load all three satellites and concatenate them into a single DataFrame, then split into train/val by randomly holding out `VAL_FRACTION` of samples.

Each sample is identified by a `gdal_vsi` path — a GDAL-compatible VSI string that allows `rasterio` to stream individual bands directly over HTTPS without downloading the entire file. Only the bands required for the current satellite are fetched at read time.


In [8]:
import pandas as pd
import rasterio
import tacoreader
from torch.utils.data import Dataset, DataLoader


class Cloud3DDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with rasterio.open(row["gdal_vsi"]) as src:
            bands = [b + 1 for b in BAND_MAPPING[row["satellite"]]]
            img   = src.read(bands).astype(np.float32)  # (15, H, W)
        d = {"image": img, "satellite": row["satellite"],
             "date": row["date"], "id": row["id"]}
        if self.transforms:
            d = self.transforms(d)
        return d


def build_dataframes(dataset_dir, val_fraction=0.05):
    dfs = []
    for sat in SATELLITES:
        f = tacoreader.load(f"{dataset_dir}{sat.lower()}/").flatten()
        dfs.append(pd.DataFrame({
            "id":        f["l0:id"],
            "satellite": f["l0:cloud3d:satellite"],
            "date":      pd.to_datetime(f["l0:stac:time_start"]),
            "gdal_vsi":  f["gdal_vsi"],
        }))
    df = pd.concat(dfs, ignore_index=True)
    val_idx    = np.random.choice(len(df), max(1, int(len(df) * val_fraction)), replace=False)
    train_mask = np.ones(len(df), dtype=bool)
    train_mask[val_idx] = False
    return df[train_mask].reset_index(drop=True), df[~train_mask].reset_index(drop=True)


transform        = build_transform(PATCH_SIZE, center_crop=True, radius=32)
train_df, val_df = build_dataframes(DATASET_DIR, VAL_FRACTION)

train_ds = Cloud3DDataset(train_df, transform)
val_ds   = Cloud3DDataset(val_df,   transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

print(f"Train samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}")
print(f"Train batches : {len(train_dl)}")

Train samples : 218628
Val   samples : 11506
Train batches : 54657



## 5. Model

The pretraining model is a **UNet autoencoder** — a symmetric encoder-decoder architecture with skip connections. It takes a `(B, 17, 128, 128)` tensor as input and outputs a `(B, 11, 128, 128)` reconstruction of the core spectral bands. The loss is pixel-wise **MSE** between the reconstruction and the original spectral channels.

We use `segmentation-models-pytorch` to build the UNet, with a lightweight **MobileNetV2** encoder trained from scratch (no ImageNet weights — geostationary imagery is spectrally very different from RGB photos).

In [9]:
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
from torch.optim import AdamW


class UNetAutoencoder(pl.LightningModule):
    def __init__(self, in_channels=N_TOTAL_BANDS, recon_channels=N_RECON_BANDS,
                 encoder_name=ENCODER, encoder_weights=None):
        super().__init__()
        self.save_hyperparameters()
        self.net = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=recon_channels,
            activation=None,
        )

    def forward(self, x):
        return self.net(x)

    def _step(self, batch):
        x = batch["image"]
        return F.mse_loss(self(x), x[:, :self.hparams.recon_channels])

    def training_step(self, batch, _):
        loss = self._step(batch)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        self.log("val_loss", self._step(batch), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

## 6. Training

We use **PyTorch Lightning** to handle the training loop, checkpointing, and logging. To keep this demo runnable on a free Colab T4 GPU, we limit each epoch to 5% of training batches and 10% of validation batches — enough to confirm the pipeline works and the loss is decreasing. Remove `limit_train_batches` and `limit_val_batches` for a full run.

Checkpoints are saved to `./output/` keeping the top-3 models by validation loss. Training metrics are logged to a CSV file for inspection after training.


In [10]:
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch import seed_everything

seed_everything(42, workers=True)
torch.backends.cudnn.benchmark     = False
torch.backends.cudnn.deterministic = True

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss", mode="min", save_top_k=3,
    save_last=True, filename="ae-{epoch:03d}-{val_loss:.4f}",
)

trainer = pl.Trainer(
    accelerator="gpu", devices=1, max_epochs=100,
    limit_train_batches=0.05, limit_val_batches=0.1,
    callbacks=[checkpoint_cb],
    logger=CSVLogger(OUTPUT_DIR, name="cloud_ae"),
)

trainer.fit(UNetAutoencoder(), train_dataloaders=train_dl, val_dataloaders=val_dl)

Epoch 0/99 ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112/2732 1:23:45 • 3:58:25 0.18it/s v_num: 0.000 train_loss_step:     
                                                                                 0.990                             

RasterioIOError: Caught RasterioIOError in DataLoader worker process 0.
Original rasterio._err.CPLE_AppDefinedError: TIFFFillTile:Read error at row 0, col 0, tile 41; got 0 bytes, expected 7751

The above exception was the direct cause of the following exception:

rasterio._err.CPLE_AppDefinedError: TIFFReadEncodedTile() failed.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "rasterio/_io.pyx", line 896, in rasterio._io.DatasetReaderBase._read
  File "rasterio/_io.pyx", line 173, in rasterio._io.io_multi_band
  File "rasterio/_io.pyx", line 179, in rasterio._io.io_multi_band
  File "rasterio/_err.pyx", line 325, in rasterio._err.StackChecker.exc_wrap_int
rasterio._err.CPLE_AppDefinedError: msg_1000km_0002D_0001R.tacozip, band 11: IReadBlock failed at X offset 1, Y offset 0: TIFFReadEncodedTile() failed.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/

## 7. Results

After training, we read the CSV log and plot train and validation loss curves to inspect convergence.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = sorted(Path(OUTPUT_DIR).glob("cloud_ae/version_*/metrics.csv"))[-1]
metrics  = pd.read_csv(log_path)

fig, ax = plt.subplots(figsize=(8, 4))
train = metrics.dropna(subset=["train_loss_epoch"])
val   = metrics.dropna(subset=["val_loss"])
ax.plot(train["epoch"], train["train_loss_epoch"], label="train loss")
ax.plot(val["epoch"],   val["val_loss"],           label="val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Cloud3D Autoencoder — Pretraining Loss")
ax.legend()
plt.tight_layout()
plt.show()